In [ ]:
!pip install datasets
!pip install tqdm

from sklearn.metrics.pairwise import cosine_similarity
from datasets import Dataset, DatasetDict, load_dataset
from huggingface_hub import login

import json
import pandas as pd
import matplotlib.pyplot as plt
import statistics
import numpy as np
import re
import seaborn as sns
from scipy.stats import norm

ORIGINAL DATASET WITH LEAST, MID AND MOST WORD ANSWERS

In [ ]:
dataset = load_dataset("Ramitha/squad1.1-600-raw-results")
df = pd.DataFrame(dataset['rawcases'])

In [30]:
cols_to_keep = [
    "ILRAlign_llama",
    "WILRAlign_llama",
    "WILRAlign_tuned_llama",
    "ILRAlign_falcon",
    "WILRAlign_falcon",
    "WILRAlign_tuned_falcon",
    "ILRAlign_gemma",
    "WILRAlign_gemma",
    "WILRAlign_tuned_gemma",
    "ILRAlign_mistral",
    "WILRAlign_mistral",
    "WILRAlign_tuned_mistral",
    "category"
]
df = df[cols_to_keep].copy()
df["gold_standard_cos"] = 1.0
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 600 entries, 0 to 599
Data columns (total 14 columns):
 #   Column                   Non-Null Count  Dtype  
---  ------                   --------------  -----  
 0   ILRAlign_llama           600 non-null    float64
 1   WILRAlign_llama          600 non-null    float64
 2   WILRAlign_tuned_llama    600 non-null    float64
 3   ILRAlign_falcon          600 non-null    float64
 4   WILRAlign_falcon         600 non-null    float64
 5   WILRAlign_tuned_falcon   600 non-null    float64
 6   ILRAlign_gemma           600 non-null    float64
 7   WILRAlign_gemma          600 non-null    float64
 8   WILRAlign_tuned_gemma    600 non-null    float64
 9   ILRAlign_mistral         600 non-null    float64
 10  WILRAlign_mistral        600 non-null    float64
 11  WILRAlign_tuned_mistral  600 non-null    float64
 12  category                 600 non-null    object 
 13  gold_standard_cos        600 non-null    float64
dtypes: float64(13), object(1)


In [31]:
results = {}

for group in ["least", "mid", "most"]:
    rows = []
    df_g = df[df["category"] == group]

    for model in ["llama", "falcon", "gemma", "mistral"]:
        for align in ["ILRAlign", "WILRAlign", "WILRAlign_tuned"]:

            mean_score = df_g[f"{align}_{model}"].mean()

            rows.append({
                "model": model,
                "alignment": align,
                "mean_alignment": round(mean_score, 4),
                "group": group
            })

    results[group] = pd.DataFrame(rows)

In [32]:
df_original_min = results["least"]
results["least"]

,model,alignment,mean_alignment,group
0,llama,ILRAlign,0.3980,least
1,llama,WILRAlign,0.4005,least
2,llama,WILRAlign_tuned,0.1573,least
3,falcon,ILRAlign,0.9673,least
4,falcon,WILRAlign,0.9673,least
5,falcon,WILRAlign_tuned,0.9467,least
6,gemma,ILRAlign,0.7686,least
7,gemma,WILRAlign,0.7688,least
8,gemma,WILRAlign_tuned,0.6272,least
9,mistral,ILRAlign,0.4892,least


In [33]:
results["mid"]

,model,alignment,mean_alignment,group
0,llama,ILRAlign,0.5680,mid
1,llama,WILRAlign,0.5706,mid
2,llama,WILRAlign_tuned,0.4894,mid
3,falcon,ILRAlign,0.9853,mid
4,falcon,WILRAlign,0.9853,mid
5,falcon,WILRAlign_tuned,0.9790,mid
6,gemma,ILRAlign,0.8880,mid
7,gemma,WILRAlign,0.8882,mid
8,gemma,WILRAlign_tuned,0.8508,mid
9,mistral,ILRAlign,0.5891,mid


In [34]:
results["most"]

,model,alignment,mean_alignment,group
0,llama,ILRAlign,0.6135,most
1,llama,WILRAlign,0.6170,most
2,llama,WILRAlign_tuned,0.5701,most
3,falcon,ILRAlign,0.9877,most
4,falcon,WILRAlign,0.9877,most
5,falcon,WILRAlign_tuned,0.9831,most
6,gemma,ILRAlign,0.9111,most
7,gemma,WILRAlign,0.9113,most
8,gemma,WILRAlign_tuned,0.8897,most
9,mistral,ILRAlign,0.6168,most


In [35]:
combined = pd.concat(
    [results["least"],
     results["mid"],
     results["most"]],
    axis=0,
    ignore_index=True
)

df_least_original = results["least"]

In [36]:
combined

,model,alignment,mean_alignment,group
0,llama,ILRAlign,0.3980,least
1,llama,WILRAlign,0.4005,least
2,llama,WILRAlign_tuned,0.1573,least
3,falcon,ILRAlign,0.9673,least
4,falcon,WILRAlign,0.9673,least
5,falcon,WILRAlign_tuned,0.9467,least
6,gemma,ILRAlign,0.7686,least
7,gemma,WILRAlign,0.7688,least
8,gemma,WILRAlign_tuned,0.6272,least
9,mistral,ILRAlign,0.4892,least


In [37]:
results = {}

for group in ["least", "mid", "most"]:
    rows = []
    df_g = df[df["category"] == group]

    for model in ["llama", "falcon", "gemma", "mistral"]:
        for align in ["ILRAlign", "WILRAlign", "WILRAlign_tuned"]:

            mean_score = df_g[f"{align}_{model}"].mean()

            rows.append({
                "model": model,
                "alignment": align,
                "mean_alignment": round(mean_score, 4),
                "group": group
            })

    results[group] = pd.DataFrame(rows)

In [38]:
summary = (
    combined
    .groupby("group")["mean_alignment"]
    .mean()
    .round(4)
)
summary

,mean_alignment
group,
least,0.6098
mid,0.7414
most,0.7725


EXPANDED 200 DATASET WITH 3 MODELS FOR LEAST AMOUNT OF WORDS

In [53]:
dataset = load_dataset("Ramitha/squad1.1-minimum-200-positive-expanded-results")
df_new = pd.DataFrame(dataset['rawcases'])
df_new = df_new[df_new["expansion_model"] != "original"]

In [54]:
df_new["category"].unique()

array(['least'], dtype=object)

In [55]:
cols_to_keep = [
    'expansion_model',
    "ILRAlign_llama",
    "WILRAlign_llama",
    "WILRAlign_tuned_llama",
    "ILRAlign_falcon",
    "WILRAlign_falcon",
    "WILRAlign_tuned_falcon",
    "ILRAlign_gemma",
    "WILRAlign_gemma",
    "WILRAlign_tuned_gemma",
    "ILRAlign_mistral",
    "WILRAlign_mistral",
    "WILRAlign_tuned_mistral"
]
df_new = df_new[cols_to_keep].copy()
df_new["category"] = 'least'
df_new["gold_standard_cos"] = 1.0

df_new.info()

<class 'pandas.core.frame.DataFrame'>
Index: 600 entries, 200 to 799
Data columns (total 15 columns):
 #   Column                   Non-Null Count  Dtype  
---  ------                   --------------  -----  
 0   expansion_model          600 non-null    object 
 1   ILRAlign_llama           600 non-null    float64
 2   WILRAlign_llama          600 non-null    float64
 3   WILRAlign_tuned_llama    600 non-null    float64
 4   ILRAlign_falcon          600 non-null    float64
 5   WILRAlign_falcon         600 non-null    float64
 6   WILRAlign_tuned_falcon   600 non-null    float64
 7   ILRAlign_gemma           600 non-null    float64
 8   WILRAlign_gemma          600 non-null    float64
 9   WILRAlign_tuned_gemma    600 non-null    float64
 10  ILRAlign_mistral         600 non-null    float64
 11  WILRAlign_mistral        600 non-null    float64
 12  WILRAlign_tuned_mistral  600 non-null    float64
 13  category                 600 non-null    object 
 14  gold_standard_cos        600 

In [56]:
results = {}

for group in ["least"]:
    rows = []
    df_g = df_new[df_new["category"] == group]

    for model in ["llama", "falcon", "gemma", "mistral"]:
        for align in ["ILRAlign", "WILRAlign", "WILRAlign_tuned"]:

            mean_score = df_g[f"{align}_{model}"].mean()

            rows.append({
                "model": model,
                "alignment": align,
                "mean_alignment": round(mean_score, 4),
                "group": group
            })

    results[group] = pd.DataFrame(rows)

combined = pd.concat(
    [results["least"]],
    axis=0,
    ignore_index=True
)
df_least_expanded = results["least"]

In [57]:
combined

,model,alignment,mean_alignment,group
0,llama,ILRAlign,0.6660,least
1,llama,WILRAlign,0.6703,least
2,llama,WILRAlign_tuned,0.6264,least
3,falcon,ILRAlign,0.9908,least
4,falcon,WILRAlign,0.9908,least
5,falcon,WILRAlign_tuned,0.9881,least
6,gemma,ILRAlign,0.9432,least
7,gemma,WILRAlign,0.9434,least
8,gemma,WILRAlign_tuned,0.9388,least
9,mistral,ILRAlign,0.6520,least


In [58]:
df_improvement = (
    df_least_original.merge(
        df_least_expanded,
        on=["model", "alignment"],
        suffixes=("_original", "_expanded")
    )
    .drop(columns=["group_original", "group_expanded"])
)

df_improvement["mean_alignment_improvement_raw"] = (
    df_improvement["mean_alignment_expanded"] -
    df_improvement["mean_alignment_original"]
)

df_improvement["mean_alignment_improvement_percent"] = (
    df_improvement["mean_alignment_improvement_raw"] /
    df_improvement["mean_alignment_original"] * 100
)

df_improvement

,model,alignment,mean_alignment_original,mean_alignment_expanded,mean_alignment_improvement_raw,mean_alignment_improvement_percent
0,llama,ILRAlign,0.3980,0.6660,0.2680,67.336683
1,llama,WILRAlign,0.4005,0.6703,0.2698,67.365793
2,llama,WILRAlign_tuned,0.1573,0.6264,0.4691,298.219962
3,falcon,ILRAlign,0.9673,0.9908,0.0235,2.429443
4,falcon,WILRAlign,0.9673,0.9908,0.0235,2.429443
5,falcon,WILRAlign_tuned,0.9467,0.9881,0.0414,4.373085
6,gemma,ILRAlign,0.7686,0.9432,0.1746,22.716628
7,gemma,WILRAlign,0.7688,0.9434,0.1746,22.710718
8,gemma,WILRAlign_tuned,0.6272,0.9388,0.3116,49.681122
9,mistral,ILRAlign,0.4892,0.6520,0.1628,33.278823


In [59]:
df_improvement_visual = df_improvement
df_improvement_visual = df_improvement_visual.drop(
    columns=[
        "mean_alignment_original",
        "mean_alignment_expanded",
        "mean_alignment_improvement_raw"
    ]
)

df_improvement_visual = df_improvement_visual.rename(
    columns={"mean_alignment_improvement_percent": "improvement %"}
)

df_improvement_visual["improvement %"] = df_improvement_visual["improvement %"].round(2)
df_improvement_visual

,model,alignment,improvement %
0,llama,ILRAlign,67.34
1,llama,WILRAlign,67.37
2,llama,WILRAlign_tuned,298.22
3,falcon,ILRAlign,2.43
4,falcon,WILRAlign,2.43
5,falcon,WILRAlign_tuned,4.37
6,gemma,ILRAlign,22.72
7,gemma,WILRAlign,22.71
8,gemma,WILRAlign_tuned,49.68
9,mistral,ILRAlign,33.28


In [60]:
original_mean = df_least_original["mean_alignment"].mean()
expanded_mean = df_least_expanded["mean_alignment"].mean()

total_improvement_percent = ((expanded_mean - original_mean) / original_mean) * 100

print("Original mean:", original_mean)
print("Expanded mean:", expanded_mean)
print("Total improvement (%):", total_improvement_percent)

Original mean: 0.6097666666666667
Expanded mean: 0.8058749999999999
Total improvement (%): 32.16120920570708


COMPARISON OF EXPANSION WITH MODELS

In [61]:
df_new.info()

<class 'pandas.core.frame.DataFrame'>
Index: 600 entries, 200 to 799
Data columns (total 15 columns):
 #   Column                   Non-Null Count  Dtype  
---  ------                   --------------  -----  
 0   expansion_model          600 non-null    object 
 1   ILRAlign_llama           600 non-null    float64
 2   WILRAlign_llama          600 non-null    float64
 3   WILRAlign_tuned_llama    600 non-null    float64
 4   ILRAlign_falcon          600 non-null    float64
 5   WILRAlign_falcon         600 non-null    float64
 6   WILRAlign_tuned_falcon   600 non-null    float64
 7   ILRAlign_gemma           600 non-null    float64
 8   WILRAlign_gemma          600 non-null    float64
 9   WILRAlign_tuned_gemma    600 non-null    float64
 10  ILRAlign_mistral         600 non-null    float64
 11  WILRAlign_mistral        600 non-null    float64
 12  WILRAlign_tuned_mistral  600 non-null    float64
 13  category                 600 non-null    object 
 14  gold_standard_cos        600 

In [62]:
df_mean = df_new.groupby("expansion_model").mean(numeric_only=True).reset_index()
alignment_columns = [
    "ILRAlign_llama","WILRAlign_llama","WILRAlign_tuned_llama",
    "ILRAlign_falcon","WILRAlign_falcon","WILRAlign_tuned_falcon",
    "ILRAlign_gemma","WILRAlign_gemma","WILRAlign_tuned_gemma",
    "ILRAlign_mistral","WILRAlign_mistral","WILRAlign_tuned_mistral"
]
df_long = df_mean.melt(
    id_vars="expansion_model",
    value_vars=alignment_columns,
    var_name="metric_model",
    value_name="mean_alignment"
)
df_long[["alignment", "model"]] = df_long["metric_model"].str.rsplit("_", n=1, expand=True)
df_result = df_long[["expansion_model", "model", "alignment", "mean_alignment"]]
df_result = df_result.sort_values(["expansion_model", "model", "alignment"]).reset_index(drop=True)
df_result

,expansion_model,model,alignment,mean_alignment
0,phi3-mini-3.8b,falcon,ILRAlign,0.992062
1,phi3-mini-3.8b,falcon,WILRAlign,0.992066
2,phi3-mini-3.8b,falcon,WILRAlign_tuned,0.990158
3,phi3-mini-3.8b,gemma,ILRAlign,0.946570
4,phi3-mini-3.8b,gemma,WILRAlign,0.946783
5,phi3-mini-3.8b,gemma,WILRAlign_tuned,0.944136
6,phi3-mini-3.8b,llama,ILRAlign,0.706949
7,phi3-mini-3.8b,llama,WILRAlign,0.712104
8,phi3-mini-3.8b,llama,WILRAlign_tuned,0.693562
9,phi3-mini-3.8b,mistral,ILRAlign,0.677349


In [63]:
df_mean = df_new.groupby("expansion_model").mean(numeric_only=True).reset_index()
alignment_columns = [
    "ILRAlign_llama","WILRAlign_llama","WILRAlign_tuned_llama",
    "ILRAlign_falcon","WILRAlign_falcon","WILRAlign_tuned_falcon",
    "ILRAlign_gemma","WILRAlign_gemma","WILRAlign_tuned_gemma",
    "ILRAlign_mistral","WILRAlign_mistral","WILRAlign_tuned_mistral"
]
df_long = df_mean.melt(
    id_vars="expansion_model",
    value_vars=alignment_columns,
    var_name="metric_model",
    value_name="mean_alignment"
)
df_long["alignment"] = df_long["metric_model"].str.replace(r"_(llama|falcon|gemma|mistral)$", "", regex=True)
df_result = (
    df_long.groupby(["expansion_model", "alignment"])["mean_alignment"]
    .mean()
    .reset_index()
    .sort_values(["expansion_model", "alignment"])
)
df_result

,expansion_model,alignment,mean_alignment
0,phi3-mini-3.8b,ILRAlign,0.830733
1,phi3-mini-3.8b,WILRAlign,0.833609
2,phi3-mini-3.8b,WILRAlign_tuned,0.818594
3,qwen-7b,ILRAlign,0.795920
4,qwen-7b,WILRAlign,0.798294
5,qwen-7b,WILRAlign_tuned,0.760638
6,smollm2-1.7b,ILRAlign,0.812347
7,smollm2-1.7b,WILRAlign,0.814788
8,smollm2-1.7b,WILRAlign_tuned,0.787935


In [64]:
df_expansion_mean = (
    df_result.groupby("expansion_model")["mean_alignment"]
    .mean()
    .reset_index()
    .rename(columns={"mean_alignment": "mean_alignment_overall"})
)

df_expansion_mean

,expansion_model,mean_alignment_overall
0,phi3-mini-3.8b,0.827645
1,qwen-7b,0.784950
2,smollm2-1.7b,0.805023


In [ ]:
baseline = 0.6097666666666667

df_expansion_mean["improvement %"] = (
    (df_expansion_mean["mean_alignment_overall"] - baseline) / baseline * 100
).round(2)

df_expansion_mean

,expansion_model,mean_alignment_overall,improvement %
0,phi3-mini-3.8b,0.827645,35.73
1,qwen-7b,0.784950,28.73
2,smollm2-1.7b,0.805023,32.02
